# Credit Manager - Database Initialization Test
Este notebook inicializa la base de datos SQLite y verifica que todas las tablas mapeadas en SQLAlchemy se hayan creado correctamente a partir de nuestro módulo `src/database`.

In [ ]:
import pandas as pd
import sys
import os

from importlib import reload

# Asegurar que el path apunte a la raíz para encontrcar el paquete 'src'
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

In [ ]:
"""
Notebook Cell: Database Reset & Modular Seeding
Description: Drops tables, recreates them, and calls the external seeding module.
Author: Juan Martín Carini
Date: 2026-05-11
"""


import src.database  # noqa: E402, F401
from src.database import SessionLocal, engine  # noqa: E402, F401
import src.database.seed_geography  # noqa: E402

def reset_and_seed():
    print("Iniciando reset de base de datos...")
    
    # 1. Limpieza total
    src.database.Base.metadata.drop_all(bind=src.database.engine)
    print("✅ Todas las tablas han sido eliminadas.")
    
    # 2. Reconstrucción del esquema
    src.database.Base.metadata.create_all(bind=src.database.engine)
    print("✅ Estructura de tablas recreada correctamente.")

    # 3. Carga de datos maestros usando el nuevo módulo
    db = src.database.SessionLocal()
    try:
        print("Poblando tablas geográficas...")
        src.database.seed_geography.seed_provincias(db)
    except Exception as e:
        db.rollback()
        print(f"❌ Error durante el seeding: {e}")
    finally:
        db.close()

if __name__ == "__main__":
    reset_and_seed()

In [ ]:
import src.database.models as models
import src.protfolio.purchase as purchase
reload(purchase)

paths = {
    "personas": "../data/PERSONAS.CSV",
    "prestamos": "../data/PRESTAMOS.CSV",
    "cuotas": "../data/CUOTAS.CSV"}

NewAssociate = models.SocioComercial()
NewAssociate.create_socio("Mentiritas S.A.", 30713257880)
NewRelation = models.Relacion()
NewRelation.add_single_mapping(1, "provincias", 7, 5)
NewRelation.add_single_mapping(1, "provincias", 1, 2)
NewRelation.add_single_mapping(1, "provincias", 3, 6)
NewAssociate = models.SocioComercial()
NewAssociate.create_socio("Mutual Uno", 30713257870)
NewAssociate.create_socio("Mutual Dos", 30713257860)
NewRelation.add_single_mapping(1, "socios_comerciales", 2, 14)
NewRelation.add_single_mapping(1, "socios_comerciales", 3, 20)
NewFolder_1 = purchase.PortfolioPurchase()
NewFolder_1.process_full_portfolio("Folder Test 1", "2026/06/03", 0.44, 30713257880, "Mentiritas S.A.", paths=None, recurso=True, iva=False)
NewFolder_2 = purchase.PortfolioPurchase()
NewFolder_2.process_full_portfolio("Folder Test 2", "2026/06/03", 0.44, 30713257880, "Mentiritas S.A.", paths=None, recurso=True, iva=False);

In [ ]:
from src.reports import saldos

df = saldos("2026/06/03", propias=True, agrupar=True, vencimientos=True)
df.loc["Total"] = df.sum()
df[["Capital", "Interés", "IVA", "Total"]] = df[["Capital", "Interés", "IVA", "Total"]].map("$ {:,.2f}".format)

df

In [ ]:
import pandas as pd  # noqa: F811
from src.database.models import Cuota, EstadoCuotaCedida, Credito, Cartera, EstadoCuota, EstadoCredito

class PortfolioSell:
    
    def __init__(self):
        """
        Initializes the portfolio sales manager with an active database session 
        and empty state containers for tracking transactional elements.
        """
        self.db = SessionLocal()
        self.cartera = None
        self.socio = None

        # State containers para la estructuración de la venta de carteras
        self.df_personas = None
        self.df_prestamos = None
        self.df_cuotas = None

    def fetch_available_installments_for_sale(self, mora: bool = True) -> pd.DataFrame:
        """
        =============================================================================
        Method: fetch_available_installments_for_sale
        Description: Queries the database for all active installments that belong 
                     to the institution and are eligible for portfolio sale. Filters 
                     out already assigned, pending, or fully cancelled records, 
                     cross-referencing current credit and installment status.
        Parameters:
            morosas (bool): If True, incorporates past-due installments and credits 
                            into the available sales pool. Defaults to True.
        Returns:
            pd.DataFrame: A DataFrame containing available installments with their 
                          associated credit and portfolio relational context.
        =============================================================================
        """
        try:
            # 1. Definir los estados admitidos en paralelo para Cuotas y Créditos
            cuotas_admitidas = [EstadoCuota.PENDIENTE]
            creditos_admitidos = [EstadoCredito.ACTIVO]

            if mora:
                cuotas_admitidas.append(EstadoCuota.MOROSA)
                creditos_admitidos.append(EstadoCredito.MOROSO)

            # 2. Construcción de la consulta con Joins para consolidar el contexto financiero
            query = (
                self.db.query(
                    Cuota.id.label("cuota_id"),
                    Cuota.nro_cuota,
                    Cuota.capital,
                    Cuota.interes,
                    # Se etiqueta de forma explícita la suma subtotal (Capital + Interés)
                    (Cuota.capital + Cuota.interes).label("subtotal"),
                    Cuota.fecha_vencimiento,
                    Credito.id.label("credito_id"),
                    Credito.cliente_cuil,
                    Cartera.id.label("cartera_origen_id"),
                    Cartera.nombre.label("cartera_origen_nombre")
                )
                .join(Credito, Cuota.credito_id == Credito.id)
                .join(Cartera, Credito.cartera_id == Cartera.id)
                .filter(
                    Credito.estado.in_(creditos_admitidos),
                    self.db.query(Cuota).filter(Cuota.estado.in_(cuotas_admitidas)).modifiers.get("dummy") is None 
                    if False else Cuota.estado.in_(cuotas_admitidas),
                    Cuota.estado_cesion == EstadoCuotaCedida.NO_VENDIDA,
                )
            )

            # 3. Ejecución y volcado directo a estructura de Pandas
            df = pd.read_sql(query.statement, self.db.get_bind())

            # 4. Normalización de formatos temporales para proyecciones de cash flow
            if not df.empty and "fecha_vencimiento" in df.columns:
                df["fecha_vencimiento"] = pd.to_datetime(df["fecha_vencimiento"]).dt.date

            return df

        except Exception as e:
            raise RuntimeError(f"Error al consultar cuotas disponibles para venta: {e}")

    def __del__(self):
        """
        Ensures the underlying SQLAlchemy connection pool drops the session 
        cleanly when the object lifecycle terminates.
        """
        if hasattr(self, "db") and self.db:
            self.db.close()

# Inicialización interactiva del manager de ventas
NewSell = PortfolioSell()
NewSell.fetch_available_installments_for_sale()